<a href="https://colab.research.google.com/github/apple-pie-h/FDS-LAB-Works/blob/main/Lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 6: Implement feature extraction and selection techniques, including experimenting with encoding methods like one-hot encoding and creating new features based on domain expertise
Feature Extraction
Feature extraction involves deriving new, more meaningful features from raw data to improve model training. Below are several feature extraction techniques:

1.1. Encoding Categorical Variables
Encoding categorical data is crucial to make it suitable for machine learning models.

a. One-Hot Encoding
Converts each category of a feature into a separate binary column.

Suitable for nominal (unordered) categorical data.
Steps:

Identify categorical columns.
Apply one-hot encoding using tools like pandas.get_dummies or OneHotEncoder from scikit-learn

In [ ]:
import pandas as pd

# Example dataset
data = {'Color': ['Red', 'Blue', 'Green', 'Blue', 'Red']}
df = pd.DataFrame(data)

# One-hot encoding
encoded_df = pd.get_dummies(df, columns=['Color'], drop_first=False)
print(encoded_df)


   Color_Blue  Color_Green  Color_Red
0       False        False       True
1        True        False      False
2       False         True      False
3        True        False      False
4       False        False       True


 Label Encoding

Assigns a unique integer to each category.

Suitable for ordinal (ordered) categorical data.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Example
label_encoder = LabelEncoder()
df['Color_Label'] = label_encoder.fit_transform(df['Color'])
print(df)


   Color  Color_Label
0    Red            2
1   Blue            0
2  Green            1
3   Blue            0
4    Red            2


Frequency Encoding

Replaces categories with their occurrence frequency.

Helps when category frequency is correlated with the target variable.

In [ ]:
frequency_map = df['Color'].value_counts().to_dict()
df['Color_Frequency'] = df['Color'].map(frequency_map)


In [ ]:
print(df)

   Color  Color_Label  Color_Frequency
0    Red            2                2
1   Blue            0                2
2  Green            1                1
3   Blue            0                2
4    Red            2                2


Handling Numerical Data

For numerical data, create new features using domain knowledge or statistical transformations.

a. Polynomial Features
Generate polynomial combinations of existing features.

In [ ]:
# Add the missing columns 'Feature1' and 'Feature2' to the DataFrame
# Example values are used, you should replace them with your actual data
import numpy as np
df['Feature1'] = np.random.rand(len(df))
df['Feature2'] = np.random.rand(len(df))

# Now you can proceed with PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
poly_features = poly.fit_transform(df[['Feature1', 'Feature2']])

In [ ]:
poly_features

array([[0.78298884, 0.54097208, 0.61307153, 0.4235751 , 0.29265079],
       [0.24479404, 0.52174518, 0.05992412, 0.12772011, 0.27221803],
       [0.43503879, 0.44583724, 0.18925874, 0.19395649, 0.19877084],
       [0.97122339, 0.27108731, 0.94327488, 0.26328634, 0.07348833],
       [0.27253177, 0.7020693 , 0.07427357, 0.19133619, 0.4929013 ]])

In [ ]:
# Add a 'Value' column to the DataFrame with example data
# Replace this with your actual data if you have it
df['Value'] = np.random.rand(len(df))

# Now calculate the rolling mean and standard deviation
df['Rolling_Mean'] = df['Value'].rolling(window=3).mean()
df['Rolling_Std'] = df['Value'].rolling(window=3).std()

In [ ]:
df

,Color,Color_Label,Color_Frequency,Feature1,Feature2,Value,Rolling_Mean,Rolling_Std
0,Red,2,2,0.782989,0.540972,0.158520,NaN,NaN
1,Blue,0,2,0.244794,0.521745,0.813701,NaN,NaN
2,Green,1,1,0.435039,0.445837,0.884138,0.618787,0.400155
3,Blue,0,2,0.971223,0.271087,0.959457,0.885765,0.072892
4,Red,2,2,0.272532,0.702069,0.962878,0.935491,0.044506


Feature Selection
Feature selection identifies the most relevant features, reducing noise and overfitting.

Filter Methods

Filter methods use statistical techniques to rank features based on their correlation with the target variable.

Correlation Analysis

Calculate the correlation coefficient between numerical features and the target variable.

In [ ]:
# Convert 'Color', 'Color_Label' and 'Color_Frequency' to numeric if possible.
# If not possible to convert, drop them before calculating correlation.
# df['Target'] = np.random.rand(len(df))

# Option 1: Drop non-numeric columns
numerical_df = df.select_dtypes(include=np.number)
correlation = numerical_df.corr()
print(correlation['Feature1'])

# Option 2: Convert 'Color_Label' and 'Color_Frequency' to numeric
# (assuming they represent ordinal or frequency data) and drop 'Color'
for col in ['Color_Label', 'Color_Frequency']:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Handle potential errors
numerical_df = df.drop(columns=['Color'])  # Drop the original 'Color' column
correlation = numerical_df.corr()
print(correlation['Feature1'])

Color_Label       -0.124616
Color_Frequency    0.184514
Feature1           1.000000
Feature2          -0.710989
Value             -0.327679
Rolling_Mean       0.152396
Rolling_Std       -0.225662
Name: Feature1, dtype: float64
Color_Label       -0.124616
Color_Frequency    0.184514
Feature1           1.000000
Feature2          -0.710989
Value             -0.327679
Rolling_Mean       0.152396
Rolling_Std       -0.225662
Name: Feature1, dtype: float64


Chi-Square Test

Used for categorical data to measure dependency between features and the target.

In [ ]:
from sklearn.feature_selection import f_regression, SelectKBest # Import f_regression
import pandas as pd
import numpy as np

# Assuming 'df' is your DataFrame and 'Target' is the target column
# If you don't have a 'Target' column, replace it with your target variable
df['Target'] = np.random.rand(len(df))
# Replace 'Target' with your target column if it exists
# and make sure 'df' contains all necessary data

# Define X and y
X = df.drop(columns=['Target', 'Color'])  # Features (excluding the target)
y = df['Target']  # Target variable

# Impute or drop NaN values in X before applying SelectKBest
# Option 1: Drop rows with NaN values
X = X.dropna()
y = y[X.index] # Ensure y is aligned with X after dropping rows

# Option 2: Impute NaN values with the mean of each column
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy='mean')
# X = imputer.fit_transform(X)

# Now apply SelectKBest with f_regression for continuous target
X_new = SelectKBest(f_regression, k=5).fit_transform(X, y) # Use f_regression
print(X_new)

[[1.         0.43503879 0.44583724 0.61878657 0.40015502]
 [0.         0.97122339 0.27108731 0.8857654  0.07289162]
 [2.         0.27253177 0.7020693  0.93549096 0.04450556]]


Wrapper Methods

Wrapper methods evaluate feature subsets by training a model and optimizing performance.

a. Recursive Feature Elimination (RFE)
Uses a model to recursively remove the least important features.
b.Forward/Backward Selection

Forward: Add features iteratively.

Backward: Remove features iteratively.

Tools: mlxtend.feature_selection.SequentialFeatureSelector.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression  # Import LinearRegression

# ... (Your existing code) ...

model = LinearRegression()  # Use LinearRegression instead of LogisticRegression
rfe = RFE(model, n_features_to_select=5)
X_rfe = rfe.fit_transform(X, y)
print(rfe.support_)

[ True  True  True  True False  True False]


Embedded Methods
Embedded methods combine feature selection with model training.

a. L1 Regularization (Lasso Regression)
Adds a penalty term to the loss function to reduce feature coefficients.

b. Tree-Based Methods
Tree-based algorithms like Random Forest or XGBoost provide feature importance scores.

 Experimentation with Feature Engineering
Experimenting with feature engineering often requires iterative testing and validation. Here's a step-by-step guide:

a. Exploratory Data Analysis (EDA)
Plot distributions to detect patterns.
Use scatterplots to examine relationships.

b. New Feature Creation
Combine existing features, e.g., interaction terms.
Generate ratios or differences: feature1 / feature2, feature1 - feature2.

c. Dimensionality Reduction
Use techniques like PCA (Principal Component Analysis) for high-dimensional data.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01)
lasso.fit(X, y)
print(lasso.coef_)


[ 0.2136726  0.        -0.         0.         0.         0.
 -0.       ]


In [ ]:
from sklearn.ensemble import RandomForestRegressor  # Import RandomForestRegressor

model = RandomForestRegressor()  # Use RandomForestRegressor for continuous target
model.fit(X, y)
importance = model.feature_importances_

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)


In [ ]:
df

,Color,Color_Label,Color_Frequency,Feature1,Feature2,Value,Rolling_Mean,Rolling_Std,Target
0,Red,2,2,0.782989,0.540972,0.158520,NaN,NaN,0.967042
1,Blue,0,2,0.244794,0.521745,0.813701,NaN,NaN,0.730303
2,Green,1,1,0.435039,0.445837,0.884138,0.618787,0.400155,0.756614
3,Blue,0,2,0.971223,0.271087,0.959457,0.885765,0.072892,0.540486
4,Red,2,2,0.272532,0.702069,0.962878,0.935491,0.044506,0.997832


Validate Feature Effectiveness

Use cross-validation to evaluate new features.

Compare model metrics like accuracy, precision, recall, or F1-score before and after adding/removing features.